<a href="https://colab.research.google.com/github/JoshuaFZ/QWEN-0.6B-LORA/blob/main/rkllm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RKLLM Colab GPU conversion

This notebook converts a HuggingFace model to RKLLM on Google Colab with GPU.

Before running it, choose Runtime, Change runtime type, GPU in Colab.


In [ ]:
from pathlib import Path
import os
# Restore configuration after Colab runtime restart.
from pathlib import Path
from datetime import datetime

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

if 'MODEL_DIR' not in globals():
    MODEL_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output/owon-qwen3-0.6b-merged')
if 'RKLLM_TOOLKIT_PACKAGE_DIR' not in globals():
    RKLLM_TOOLKIT_PACKAGE_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/rkllm-toolkit/packages')
if 'RKLLM_TOOLKIT_WHEEL' not in globals():
    RKLLM_TOOLKIT_WHEEL = None
if 'DATA_QUANT' not in globals():
    DATA_QUANT = MODEL_DIR / 'data_quant.json'
if 'JSONL_DATASET' not in globals():
    JSONL_DATASET = Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl')
if 'MAX_CALIBRATION_ROWS' not in globals():
    MAX_CALIBRATION_ROWS = None
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = MODEL_DIR
if 'TARGET_PLATFORM' not in globals():
    TARGET_PLATFORM = 'RK3588'
if 'QUANTIZED_DTYPE' not in globals():
    QUANTIZED_DTYPE = 'W8A8'
if 'QUANTIZED_ALGORITHM' not in globals():
    QUANTIZED_ALGORITHM = 'normal'
if 'OPTIMIZATION_LEVEL' not in globals():
    OPTIMIZATION_LEVEL = 0
if 'NUM_NPU_CORE' not in globals():
    NUM_NPU_CORE = 3
if 'MAX_CONTEXT' not in globals():
    MAX_CONTEXT = 4096
if 'LOAD_DTYPE' not in globals():
    LOAD_DTYPE = 'float32'
if 'OUTPUT_PATH' not in globals():
    DATE_TAG = datetime.now().strftime('%Y%m%d')
    OUTPUT_PATH = OUTPUT_DIR / f'{MODEL_DIR.name}_{QUANTIZED_DTYPE}_{TARGET_PLATFORM}_{DATE_TAG}.rkllm'
print('MODEL_DIR =', MODEL_DIR)
print('DATA_QUANT =', DATA_QUANT)
print('OUTPUT_PATH =', OUTPUT_PATH)


os.environ['USE_TORCH'] = '1'
os.environ['USE_JAX'] = '0'
# Disable TensorFlow/Flax backends in transformers. RKLLM conversion only needs PyTorch.
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

from rkllm.api import RKLLM
if not DATA_QUANT.exists():
    quant_candidates = [
        MODEL_DIR / 'data_quant.json',
        Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output/owon-qwen3-0.6b-merged/data_quant.json'),
    ]
    quant_candidates = [p for p in quant_candidates if p.exists()]
    if quant_candidates:
        DATA_QUANT = quant_candidates[0]
        print('Using discovered quant data:', DATA_QUANT)

for path in [MODEL_DIR, DATA_QUANT, OUTPUT_DIR]:
    if not Path(path).exists():
        raise FileNotFoundError(path)

llm = RKLLM()

print('Loading HuggingFace model on CUDA...')
ret = llm.load_huggingface(
    model=str(MODEL_DIR),
    model_lora=None,
    device='cuda',
    dtype=LOAD_DTYPE,
    custom_config=None,
    load_weight=True,
)
if ret != 0:
    raise RuntimeError(f'Load model failed: {ret}')

print('Building RKLLM...')
ret = llm.build(
    do_quantization=True,
    optimization_level=OPTIMIZATION_LEVEL,
    quantized_dtype=QUANTIZED_DTYPE,
    quantized_algorithm=QUANTIZED_ALGORITHM,
    target_platform=TARGET_PLATFORM,
    num_npu_core=NUM_NPU_CORE,
    extra_qparams=None,
    dataset=str(DATA_QUANT),
    hybrid_rate=0,
    max_context=MAX_CONTEXT,
)
if ret != 0:
    raise RuntimeError(f'Build model failed: {ret}')

print('Exporting:', OUTPUT_PATH)
ret = llm.export_rkllm(str(OUTPUT_PATH))
if ret != 0:
    raise RuntimeError(f'Export model failed: {ret}')

print('Conversion succeeded:', OUTPUT_PATH)
print('File size MB:', OUTPUT_PATH.stat().st_size / 1024 / 1024)



## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Configure paths and build options

Change these paths to match your Google Drive layout. Put the rkllm-toolkit wheel files from rkllm-toolkit/packages into Drive, or set RKLLM_TOOLKIT_WHEEL directly.


In [ ]:
from pathlib import Path
from datetime import datetime

MODEL_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output/owon-qwen3-0.6b-merged')
RKLLM_TOOLKIT_PACKAGE_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/rkllm-toolkit/packages')
RKLLM_TOOLKIT_WHEEL = None

DATA_QUANT = MODEL_DIR / 'data_quant.json'
JSONL_DATASET = Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl')
MAX_CALIBRATION_ROWS = None

OUTPUT_DIR = MODEL_DIR
TARGET_PLATFORM = 'RK3588'
QUANTIZED_DTYPE = 'W8A8'
QUANTIZED_ALGORITHM = 'normal'
OPTIMIZATION_LEVEL = 0
NUM_NPU_CORE = 3
MAX_CONTEXT = 4096
LOAD_DTYPE = 'float32'

DATE_TAG = datetime.now().strftime('%Y%m%d')
OUTPUT_PATH = OUTPUT_DIR / f'{MODEL_DIR.name}_{QUANTIZED_DTYPE}_{TARGET_PLATFORM}_{DATE_TAG}.rkllm'
print('MODEL_DIR =', MODEL_DIR)
print('DATA_QUANT =', DATA_QUANT)
print('OUTPUT_PATH =', OUTPUT_PATH)



## 3a. Install dependencies, then restart runtime

This cell installs the dependency versions expected by rkllm-toolkit. It downloads a large CUDA PyTorch stack, so it can take several minutes. Dependency conflict warnings from preinstalled Colab packages are expected. After successful installation, the cell forcibly restarts the runtime to avoid NumPy binary ABI conflicts. After Colab reconnects, run the next cell, not this one again.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
DEPS_MARKER = Path(f'/content/.rkllm_colab_deps_installed_{py_tag}')


def run(cmd):
    print('+', ' '.join(map(str, cmd)), flush=True)
    subprocess.check_call(list(map(str, cmd)))


if not DEPS_MARKER.exists():
    run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])

    # Install the dependency set needed by RKLLM conversion. Do not install auto_gptq here:
    # auto_gptq 0.7.1 is often unavailable/incompatible on Colab Python 3.12.
    run([
        sys.executable, '-m', 'pip', 'install', '-U',
        'numpy<=1.26.4,>=1.23.1',
        'torch==2.6.0',
        'torchvision==0.21.0',
        'transformers==4.55.2',
        'datasets==4.1.1',
        'pyarrow==21.0.0',
        'accelerate<=1.5.2,>=1.0.1',
        'sentencepiece==0.2.0',
        'protobuf<=4.25.4,>=4.21.6',
        'transformers_stream_generator==0.0.5',
        'einops==0.4.1',
        'scipy>=1.5.4',
        'tiktoken<=0.9.0,>=0.7.0',
        'tabulate==0.9.0',
        'Jinja2==3.1.4',
        'safetensors==0.5.3',
        'colorlog==6.8.2',
        'datamodel_code_generator==0.26.0',
        'jsonschema==4.23.0',
        'flatbuffers==24.3.25',
        'pillow==10.4.0',
        'optimum==1.23.3',
        'jsonlines==4.0.0',
        'timm==1.0.19',
    ])

    DEPS_MARKER.write_text('ok', encoding='utf-8')
    print('Dependencies installed. Restarting runtime now to avoid NumPy ABI conflicts...', flush=True)
    print('After Colab reconnects, run the next cell: 3b. Install rkllm-toolkit and verify.', flush=True)
    os.kill(os.getpid(), 9)

print('Dependency marker found:', DEPS_MARKER)
print('Skip dependency installation. Run the next cell if RKLLM is not verified yet.')



## 3b. Install rkllm-toolkit wheel and verify CUDA

Run this after the runtime restarts from step 3a. The cell searches Drive, /content, and a cloned RKNN-LLM repo for the matching rkllm-toolkit wheel. The wheel is installed with --no-deps because its metadata includes optional/problematic packages such as auto_gptq.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/airockchip/rknn-llm.git'
REPO_DIR = Path('/content/rknn-llm')

# Colab runtime restart clears variables from previous cells. Provide safe defaults.
RKLLM_TOOLKIT_WHEEL = globals().get('RKLLM_TOOLKIT_WHEEL', None)
RKLLM_TOOLKIT_PACKAGE_DIR = Path(globals().get(
    'RKLLM_TOOLKIT_PACKAGE_DIR',
    '/content/drive/MyDrive/rknn-llm/rkllm-toolkit/packages',
))
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
RKLLM_MARKER = Path(f'/content/.rkllm_toolkit_installed_{py_tag}')


def run(cmd):
    print('+', ' '.join(map(str, cmd)), flush=True)
    subprocess.check_call(list(map(str, cmd)))


def find_rkllm_wheel():
    wheel_pattern = f'rkllm_toolkit-*-{py_tag}-{py_tag}-linux_x86_64.whl'

    wheel = RKLLM_TOOLKIT_WHEEL
    if wheel is not None:
        wheel = Path(wheel)
        if wheel.exists():
            return wheel

    candidates = []
    search_dirs = [RKLLM_TOOLKIT_PACKAGE_DIR, Path('/content'), Path('/content/drive/MyDrive')]
    for search_dir in search_dirs:
        if search_dir.exists():
            candidates.extend(sorted(search_dir.rglob(wheel_pattern)))
    if candidates:
        return candidates[-1]

    print('No local rkllm-toolkit wheel found. Cloning RKNN-LLM repo to /content...', flush=True)
    if not REPO_DIR.exists():
        run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    candidates = sorted(REPO_DIR.rglob(wheel_pattern))
    if candidates:
        return candidates[-1]

    print('Still no matching wheel found after Drive search and git clone.')
    print('Current Python requires:', wheel_pattern)
    print('You can upload the matching wheel now. For this repo, use:')
    print('rkllm-toolkit/packages/rkllm_toolkit-1.2.2-' + py_tag + '-' + py_tag + '-linux_x86_64.whl')
    try:
        from google.colab import files
        uploaded = files.upload()
        uploaded_candidates = [Path(name) for name in uploaded if Path(name).match(wheel_pattern)]
        if uploaded_candidates:
            return uploaded_candidates[-1]
    except Exception as exc:
        print('Upload helper is unavailable:', exc)

    raise FileNotFoundError(
        'No matching rkllm-toolkit wheel was found.\n'
        f'Current Python requires: {wheel_pattern}\n'
        f'Default expected directory: {RKLLM_TOOLKIT_PACKAGE_DIR}\n'
        'Copy/upload the wheel, or set RKLLM_TOOLKIT_WHEEL to the exact .whl path.'
    )


if not RKLLM_MARKER.exists():
    wheel = find_rkllm_wheel()
    print('Using rkllm-toolkit wheel:', wheel)
    run([sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', str(wheel)])
    RKLLM_MARKER.write_text(str(wheel), encoding='utf-8')
else:
    print('RKLLM toolkit marker found:', RKLLM_MARKER)
    print('Installed wheel:', RKLLM_MARKER.read_text(encoding='utf-8').strip())

BACKEND_CLEAN_MARKER = Path(f'/content/.rkllm_tf_jax_removed_{py_tag}')
if not BACKEND_CLEAN_MARKER.exists():
    print('Removing preinstalled TF/JAX/Flax backends...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'tensorflow', 'tensorflow-cpu', 'tensorflow-probability', 'tensorflow-datasets', 'keras', 'jax', 'jaxlib', 'flax'], check=False)
    BACKEND_CLEAN_MARKER.write_text('ok', encoding='utf-8')
    print('Backend cleanup done. Restarting runtime to clear import caches...', flush=True)
    print('After Colab reconnects, rerun this same 3b cell.', flush=True)
    os.kill(os.getpid(), 9)
print('Backend cleanup marker found:', BACKEND_CLEAN_MARKER)
# Disable TensorFlow/Flax backends in transformers. RKLLM conversion only needs PyTorch.
os.environ['USE_TF'] = '0'
os.environ['USE_TORCH'] = '1'
os.environ['USE_JAX'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
for module_name in list(sys.modules):
    if module_name == 'transformers' or module_name.startswith(('transformers.', 'tensorflow', 'jax', 'flax')):
        sys.modules.pop(module_name, None)

import numpy as np
import torch
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. In Colab, select Runtime, Change runtime type, GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from rkllm.api import RKLLM
print('RKLLM toolkit import OK')






## 4.** Prepare quantization data**

If DATA_QUANT already exists, it is reused. Otherwise the notebook generates it from JSONL_DATASET using an Alpaca-style prompt.


In [ ]:
import json
# Restore configuration after Colab runtime restart.
from pathlib import Path
from datetime import datetime

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

if 'MODEL_DIR' not in globals():
    MODEL_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output/owon-qwen3-0.6b-merged')
if 'RKLLM_TOOLKIT_PACKAGE_DIR' not in globals():
    RKLLM_TOOLKIT_PACKAGE_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/rkllm-toolkit/packages')
if 'RKLLM_TOOLKIT_WHEEL' not in globals():
    RKLLM_TOOLKIT_WHEEL = None
if 'DATA_QUANT' not in globals():
    DATA_QUANT = MODEL_DIR / 'data_quant.json'
if 'JSONL_DATASET' not in globals():
    JSONL_DATASET = Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl')
if 'REGENERATE_DATA_QUANT' not in globals():
    REGENERATE_DATA_QUANT = True
if 'MAX_CALIBRATION_ROWS' not in globals():
    MAX_CALIBRATION_ROWS = None
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = MODEL_DIR
if 'TARGET_PLATFORM' not in globals():
    TARGET_PLATFORM = 'RK3588'
if 'QUANTIZED_DTYPE' not in globals():
    QUANTIZED_DTYPE = 'W8A8'
if 'QUANTIZED_ALGORITHM' not in globals():
    QUANTIZED_ALGORITHM = 'normal'
if 'OPTIMIZATION_LEVEL' not in globals():
    OPTIMIZATION_LEVEL = 0
if 'NUM_NPU_CORE' not in globals():
    NUM_NPU_CORE = 3
if 'MAX_CONTEXT' not in globals():
    MAX_CONTEXT = 4096
if 'LOAD_DTYPE' not in globals():
    LOAD_DTYPE = 'float32'
if 'OUTPUT_PATH' not in globals():
    DATE_TAG = datetime.now().strftime('%Y%m%d')
    OUTPUT_PATH = OUTPUT_DIR / f'{MODEL_DIR.name}_{QUANTIZED_DTYPE}_{TARGET_PLATFORM}_{DATE_TAG}.rkllm'
print('MODEL_DIR =', MODEL_DIR)
print('DATA_QUANT =', DATA_QUANT)
print('OUTPUT_PATH =', OUTPUT_PATH)


PROMPT_TEMPLATE = (
    'Below is an instruction that describes a task, paired with an input that provides further context. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n\n'
    '### Input:\n{input}\n\n'
    '### Response:'
)

def build_data_quant(jsonl_path, output_path, limit=None):
    rows = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            target = record.get('output', '')
            if not isinstance(target, str):
                target = json.dumps(target, ensure_ascii=False, separators=(',', ':'))
            rows.append({
                'input': PROMPT_TEMPLATE.format(instruction=record.get('instruction', ''), input=record.get('input', '')),
                'target': target,
            })
            if limit and len(rows) >= limit:
                break
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    return len(rows)

if not DATA_QUANT.exists():
    quant_candidates = [
        MODEL_DIR / 'data_quant.json',
        Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output/owon-qwen3-0.6b-merged/data_quant.json'),
    ]
    quant_candidates = [p for p in quant_candidates if p.exists()]
    if quant_candidates:
        DATA_QUANT = quant_candidates[0]
        print('Using discovered quant data:', DATA_QUANT)
if not DATA_QUANT.exists() and not JSONL_DATASET.exists():
    jsonl_candidates = [
        Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl'),
        MODEL_DIR.parent.parent / 'LORA_train-qwen0.6B.jsonl',
    ]
    jsonl_candidates = [p for p in jsonl_candidates if p.exists()]
    if jsonl_candidates:
        JSONL_DATASET = jsonl_candidates[0]
        print('Using JSONL dataset:', JSONL_DATASET)


if REGENERATE_DATA_QUANT and JSONL_DATASET.exists():
    n = build_data_quant(JSONL_DATASET, DATA_QUANT, MAX_CALIBRATION_ROWS)
    print(f'Regenerated quant data: {DATA_QUANT} ({n} records)')
elif DATA_QUANT.exists():
    with open(DATA_QUANT, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    print(f'Using existing quant data: {DATA_QUANT} ({len(rows)} records)')
else:
    raise FileNotFoundError(f'DATA_QUANT not found and JSONL_DATASET does not exist: {JSONL_DATASET}')

## 5. Convert with GPU and export RKLLM

The output filename includes the date, for example model_W8A8_RK3588_20260630.rkllm.


In [ ]:
from pathlib import Path
import os
# Restore configuration after Colab runtime restart.
from pathlib import Path
from datetime import datetime

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

if 'MODEL_DIR' not in globals():
    MODEL_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output/owon-qwen3-0.6b-merged')
if 'RKLLM_TOOLKIT_PACKAGE_DIR' not in globals():
    RKLLM_TOOLKIT_PACKAGE_DIR = Path('/content/drive/MyDrive/qwen3-0.6B/rkllm-toolkit/packages')
if 'RKLLM_TOOLKIT_WHEEL' not in globals():
    RKLLM_TOOLKIT_WHEEL = None
if 'DATA_QUANT' not in globals():
    DATA_QUANT = MODEL_DIR / 'data_quant.json'
if 'JSONL_DATASET' not in globals():
    JSONL_DATASET = Path('/content/drive/MyDrive/qwen3-0.6B/LORA_train-qwen0.6B.jsonl')
if 'REGENERATE_DATA_QUANT' not in globals():
    REGENERATE_DATA_QUANT = True
if 'MAX_CALIBRATION_ROWS' not in globals():
    MAX_CALIBRATION_ROWS = None
if 'OUTPUT_DIR' not in globals():
    OUTPUT_DIR = MODEL_DIR
if 'TARGET_PLATFORM' not in globals():
    TARGET_PLATFORM = 'RK3588'
if 'QUANTIZED_DTYPE' not in globals():
    QUANTIZED_DTYPE = 'W8A8'
if 'QUANTIZED_ALGORITHM' not in globals():
    QUANTIZED_ALGORITHM = 'normal'
if 'OPTIMIZATION_LEVEL' not in globals():
    OPTIMIZATION_LEVEL = 0
if 'NUM_NPU_CORE' not in globals():
    NUM_NPU_CORE = 3
if 'MAX_CONTEXT' not in globals():
    MAX_CONTEXT = 4096
if 'LOAD_DTYPE' not in globals():
    LOAD_DTYPE = 'float32'
if 'OUTPUT_PATH' not in globals():
    DATE_TAG = datetime.now().strftime('%Y%m%d')
    OUTPUT_PATH = OUTPUT_DIR / f'{MODEL_DIR.name}_{QUANTIZED_DTYPE}_{TARGET_PLATFORM}_{DATE_TAG}.rkllm'
print('MODEL_DIR =', MODEL_DIR)
print('DATA_QUANT =', DATA_QUANT)
print('OUTPUT_PATH =', OUTPUT_PATH)


os.environ['USE_TORCH'] = '1'
os.environ['USE_JAX'] = '0'
# Disable TensorFlow/Flax backends in transformers. RKLLM conversion only needs PyTorch.
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

from rkllm.api import RKLLM
if not DATA_QUANT.exists():
    quant_candidates = [
        MODEL_DIR / 'data_quant.json',
        Path('/content/drive/MyDrive/qwen3-0.6B/owon-qwen3-0.6b-output/owon-qwen3-0.6b-merged/data_quant.json'),
    ]
    quant_candidates = [p for p in quant_candidates if p.exists()]
    if quant_candidates:
        DATA_QUANT = quant_candidates[0]
        print('Using discovered quant data:', DATA_QUANT)

for path in [MODEL_DIR, DATA_QUANT, OUTPUT_DIR]:
    if not Path(path).exists():
        raise FileNotFoundError(path)

llm = RKLLM()

print('Loading HuggingFace model on CUDA...')
ret = llm.load_huggingface(
    model=str(MODEL_DIR),
    model_lora=None,
    device='cuda',
    dtype=LOAD_DTYPE,
    custom_config=None,
    load_weight=True,
)
if ret != 0:
    raise RuntimeError(f'Load model failed: {ret}')

print('Building RKLLM...')
ret = llm.build(
    do_quantization=True,
    optimization_level=OPTIMIZATION_LEVEL,
    quantized_dtype=QUANTIZED_DTYPE,
    quantized_algorithm=QUANTIZED_ALGORITHM,
    target_platform=TARGET_PLATFORM,
    num_npu_core=NUM_NPU_CORE,
    extra_qparams=None,
    dataset=str(DATA_QUANT),
    hybrid_rate=0,
    max_context=MAX_CONTEXT,
)
if ret != 0:
    raise RuntimeError(f'Build model failed: {ret}')

print('Exporting:', OUTPUT_PATH)
ret = llm.export_rkllm(str(OUTPUT_PATH))
if ret != 0:
    raise RuntimeError(f'Export model failed: {ret}')

print('Conversion succeeded:', OUTPUT_PATH)
print('File size MB:', OUTPUT_PATH.stat().st_size / 1024 / 1024)


## 6. Optional local download

If OUTPUT_PATH points to /content instead of Drive, uncomment this cell to download the file.


In [ ]:
# from google.colab import files
# files.download(str(OUTPUT_PATH))
